# Logistische Regression (Optimierung der negativen LogLikelihood-Funktion)

In [134]:
%pip install datasets

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [135]:
from datasets import load_dataset
dataset = load_dataset("sms_spam")
dataset

DatasetDict({
    train: Dataset({
        features: ['sms', 'label'],
        num_rows: 5574
    })
})

In [136]:
dataset["train"]["sms"][2]

"Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's\n"

In [137]:
dataset["train"]["label"][2]

1

In [138]:
len(dataset["train"]["sms"]) # 5574 SMS 

5574

$x \in \mathbb{R}^d$

In [139]:
dataset["train"]["sms"][2]

"Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's\n"

In [140]:
dataset["train"]["sms"][9]

'Had your mobile 11 months or more? U R entitled to Update to the latest colour mobiles with camera for Free! Call The Mobile Update Co FREE on 08002986030\n'

1. ASCII
    - einfach, Alphabet ist klein (256 Zeichen)
    - Nachteil: sehr lange Sequenzen 
2. Tokenisierung
    - " Up" -> 10
    - "date " -> 22 
    - Update = [10,22]
    - Nachteil: immer noch eine Sequenz 
3. Vektorisierung
    - ganze Wörter als Einträge in den Vektor

In [141]:
doc1 = "large string"
doc2 = "second large string"
doc3 = "third very very large string"

| x | large | string | second | third | very | Typ |
|---|-------|--------|--------|-------|------|-----|
| doc1 | 1  |   1    |    0   |   0   |  0   |  binary |
| doc3 | 1   |  1    |    0   |    1  |  2   |  count |
| doc3 | 1   |  1    |    0   |    1  |  1   |  binary |

In [142]:
from sklearn.model_selection import train_test_split

In [143]:
X = dataset["train"].train_test_split(test_size=0.5) # 80/20 split 

In [144]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(max_features=1000,binary=True) # 1000 häufigste Wörter
X_train = vectorizer.fit_transform(x["sms"] for x in X["train"]).toarray()
X_test = vectorizer.transform(x["sms"] for x in X["test"]).toarray()
y_train = [x["label"] for x in X["train"]]
y_test = [x["label"] for x in X["test"]]

In [145]:
# Data Leakage 
# dataset["train"] # ursprünglicher Datensatz
# vectorizer = CountVectorizer(max_features=1000,binary=True)
# X_falsch = vectorizer.fit_transform(x["sms"] for x in dataset["train"]).toarray()
# X_train_f, X_test_f = train_test_split(X_falsch, test_size=0.2)

In [146]:
import numpy as np

X_train_np = X_train.astype(np.float32)
X_test_np = X_test.astype(np.float32)
y_train_np = np.asarray(y_train, dtype=np.float32)
y_test_np = np.asarray(y_test, dtype=np.float32)

# z = b*1 + w1*x1 + ... + wd*xd
# Bias 
X_train_b = np.hstack([np.ones((X_train_np.shape[0],1)), X_train_np])
X_test_b = np.hstack([np.ones((X_test_np.shape[0],1)), X_test_np])
X_train_b

array([[1., 0., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       ...,
       [1., 0., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.]], shape=(2787, 1001))

In [147]:
len(X_train_np)

2787

In [148]:
def sigmoid(z):
    return 1/(1+np.exp(-z))

alpha = 0.1 # Lernrate 
n_epochs = 200 # 1 Epoche = 1 Mal hat ML Algorithmus den gesamten Datensatz "gesehen"
batch_size = 64 

n = len(X_train_b) 
w = np.random.randn(X_train_b.shape[1],1) # irgendwo zufällig 

w_start = w 
for epoch in range(n_epochs): # Training-Loop 
    shuffle_idx = np.random.permutation(n) # Mischen 
    X_b_shuffle = X_train_b[shuffle_idx]
    y_shuffle = y_train_np[shuffle_idx]
    for i in range(0, n - batch_size, batch_size): # von Null bis N in batch_size Schritten 
        xi = X_b_shuffle[i:i+batch_size] # Auswahl vom Datenpunkten 
        yi = y_shuffle[i:i+batch_size]
        # z = b*1 + w1*x1 + ... + wd*xd 
        logits = xi @ w # lineares Modell, logit = Rohwert vor der Aktivierung  
        probs = sigmoid(logits)
        
        gradients = xi.T @ (probs.T - yi).T
        w = w - alpha * gradients

In [149]:
test_probs = sigmoid(X_test_b @ w)
test_pred = (test_probs > 0.5).astype(np.int32)
acc = (test_pred.T == y_test_np).mean() # Genauigkeit
print(f"Genauigkeit: {acc*100:.2f}%")

Genauigkeit: 98.53%


In [150]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(test_pred, y_test_np)
cm

array([[2399,   27],
       [  14,  347]])

In [151]:
def predict_spam(msg): 
    x = vectorizer.transform([msg]).toarray().astype(np.float32)
    x_b = np.hstack([np.ones((x.shape[0], 1), dtype=np.float32), x])
    prob = sigmoid(x_b @ w)[0]
    return "SPAM" if prob > 0.5 else "NO SPAM" 


print(predict_spam('Congratulations, you have won a prize!'))
print(predict_spam('Are we meeting at 7pm for dinner?'))

NO SPAM
NO SPAM


In [152]:
w

array([[-6.75347198],
       [ 1.08396602],
       [-0.3460059 ],
       ...,
       [ 0.05937091],
       [ 1.01337432],
       [-1.39705814]], shape=(1001, 1))

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

X_train_torch = torch.tensor(X_train, dtype=torch.float32)
y_train_torch = torch.tensor(y_train, dtype=torch.float32)
X_test_torch = torch.tensor(X_test, dtype=torch.float32)
y_test_torch = torch.tensor(y_test, dtype=torch.float32)

train_dataset = TensorDataset(X_train_torch, y_train_torch.unsqueeze(1))
test_dataset = TensorDataset(X_test_torch, y_test_torch.unsqueeze(1))


train_loader = DataLoader(train_dataset, batch_size= 64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=X_test_torch.shape[0], shuffle=False)


class LogisticRegressor(nn.Module): 
    def __init__(self, input_size):
        super().__init__() # Vererbung aus der Superklasse
        self.linear = nn.Linear(input_size,  1) 
        self.sigmoid = nn.Sigmoid()
    
    def forward(self,x): # durch das Netz die Daten propagieren 
        return self.sigmoid(self.linear(x))

# Feed-Forward Neural Network 
class NNSMSClassifier(nn.Module): 
    def __init__(self, input_size):
        super().__init__() # Vererbung aus der Superklasse
        self.input = nn.Linear(input_size, 64 )
        self.hidden = nn.Linear(64, 1)  
        self.sigmoid = nn.Sigmoid()
    
    def forward(self,x): # durch das Netz die Daten propagieren 
        x = self.input(x)
        x = torch.relu(x) # ReLU-Aktivierung
        x = self.hidden(x) 
        x = self.sigmoid(x) # Entscheidung ob 0 oder 1 
        return x 


model = NNSMSClassifier(input_size=X_train.shape[1])
loss_fn = nn.BCELoss() # Binary Cross Entropy ( =Negative LogLikelihood)
optimizer = torch.optim.SGD(model.parameters(), lr = 0.1)

n_epochs = 50

for epoch in range(n_epochs):
    for xb,yb in train_loader:
        probs = model(xb) # Wahrscheinlichkeiten für SMS (SPAM/NO SPAM)
        loss = loss_fn(probs, yb) # Unterschied Vorhersage / Realität 

        optimizer.zero_grad()
        loss.backward() # Gradienten
        optimizer.step()

model.eval() # Versetzung ins Evaluierungsmodus 
with torch.no_grad():
    probs_test = model(X_test_torch)
    preds_test = (probs_test > 0.5).cpu().numpy().astype(np.int32).ravel()


acc = (preds_test == y_test).mean()
print(f"Genauigkeit: {acc*100:.2f}%")

Genauigkeit: 98.49%


In [156]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(preds_test, y_test)
cm

array([[2406,   35],
       [   7,  339]])